# MCA GPU Evaluator (Kaggle, 2× T4 / P100)

Headless batch benchmark: **raw LLMs vs. MCA-wrapped LLMs** on the adversarial metacognition set
`Mati83moni/mca-adversarial-benchmarks` (paradoxes · false premises · missing data · control facts).

Outputs (in `/kaggle/working/mca_results`): per-item JSONL, metrics JSON, comparison table,
confusion matrices, **False Acceptance Rate (FAR)**, `FAR_abstain`, over-refusal, ECE and
**Paradox Identification Accuracy**.

Settings → Accelerator: *GPU T4 x2* (or P100). Internet: **on**. Add a Kaggle secret `HF_TOKEN`
for gated models (Llama-3) and for optional result upload.

In [ ]:
%%capture
!pip install -q "z3-solver>=4.12" "transformers>=4.44" "accelerate>=0.33" sentencepiece "huggingface_hub>=0.24" bitsandbytes
# MCA framework (set MCA_REF to a branch/tag/commit)
MCA_REPO, MCA_REF = "https://github.com/Mati83mon/MCA.git", "main"
!pip install -q "git+{MCA_REPO}@{MCA_REF}"

In [ ]:
import gc, json, os, time, asyncio
from pathlib import Path
import torch

print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory / 2**30:.1f} GiB")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as exc:
    print("No HF_TOKEN secret (gated models / upload disabled):", type(exc).__name__)

OUT = Path("/kaggle/working/mca_results"); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download
from benchmarks.runner import load_questions

DATASET_REPO = "Mati83moni/mca-adversarial-benchmarks"
path = hf_hub_download(DATASET_REPO, "data/pytania_podchwytliwe.jsonl", repo_type="dataset")
ITEMS = load_questions(path)          # Pydantic-validated QuestionItem objects
print(len(ITEMS), "questions;", {c: sum(i.kategoria.value == c for i in ITEMS) for c in sorted({i.kategoria.value for i in ITEMS})})

In [ ]:
N_GPU = torch.cuda.device_count()
SMALL_VRAM = N_GPU == 1 and torch.cuda.get_device_properties(0).total_memory < 20 * 2**30   # P100 / single T4

MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Meta-Llama-3-8B-Instruct",        # gated: needs HF_TOKEN with access
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
]
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"   # Polish-capable NLI
LOAD_IN_4BIT = SMALL_VRAM            # 7-8B fp16 fits only when sharded over 2×T4
NLI_DEVICE = f"cuda:{N_GPU - 1}" if N_GPU else "cpu"
MAX_NEW_TOKENS = 256
LIMIT = None                         # e.g. 20 for a smoke run
items = ITEMS[:LIMIT] if LIMIT else ITEMS

In [ ]:
from benchmarks.runner import _evaluate_raw, _evaluate_mca, summarize
from mca import MCAConfig, MCAOrchestrator
from mca.core.evidence_verifier import EvidenceStore
from mca.llm.transformers_llm import TransformersLLM
from mca.utils.embeddings import HashingEmbedder
from mca.utils.nli_client import NLIClient, TransformersNLIBackend

nli = NLIClient(TransformersNLIBackend(NLI_MODEL, device=NLI_DEVICE))
store = EvidenceStore.from_seed(HashingEmbedder())       # thresholds are calibrated for this embedder
cfg = MCAConfig(nli_backend="transformers", nli_model=NLI_MODEL, embedding_backend="hashing",
                n_samples=5, max_new_tokens=MAX_NEW_TOKENS)

def save(tag, rows, metrics):
    with open(OUT / f"{tag}.jsonl", "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    (OUT / f"{tag}.metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2))

ALL = {}
for model_name in MODELS:
    tag = model_name.split("/")[-1]
    try:
        llm = TransformersLLM(model_name, device_map="auto", load_in_4bit=LOAD_IN_4BIT)
    except Exception as exc:
        hint = " -> accept the model licence on its HF page for the account behind HF_TOKEN" if "gated" in str(exc) else ""
        print(f"skip {model_name}: {type(exc).__name__}: {str(exc)[:200]}{hint}")
        continue
    t0 = time.time()
    raw_rows = await _evaluate_raw(llm, items, MAX_NEW_TOKENS)
    ALL[f"raw/{tag}"] = summarize(raw_rows); save(f"raw__{tag}", raw_rows, ALL[f"raw/{tag}"])
    orch = MCAOrchestrator(llm, cfg, store=store, nli=nli)
    mca_rows = await _evaluate_mca(orch, items, 1)
    orch.close()
    ALL[f"mca/{tag}"] = summarize(mca_rows); save(f"mca__{tag}", mca_rows, ALL[f"mca/{tag}"])
    print(f"{tag}: raw acc={ALL[f'raw/{tag}']['accuracy']:.3f} FAR={ALL[f'raw/{tag}']['FAR']:.3f} | "
          f"MCA acc={ALL[f'mca/{tag}']['accuracy']:.3f} FAR={ALL[f'mca/{tag}']['FAR']:.3f}  ({time.time() - t0:.0f}s)")
    del llm, orch; gc.collect(); torch.cuda.empty_cache()

if not ALL:
    raise RuntimeError("No model could be loaded - see the skip messages above")


In [ ]:
import pandas as pd
cols = ["accuracy", "macro_f1", "FAR", "FAR_abstain", "over_refusal", "paradox_identification_accuracy", "ECE", "latency_ms_p50"]
table = pd.DataFrame({k: {c: v.get(c) for c in cols} for k, v in ALL.items()}).T
table.to_csv(OUT / "comparison.csv")
table.style.format(precision=3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

keys = list(ALL)
fig, axes = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 4), squeeze=False)
for ax, key in zip(axes[0], keys):
    m = np.array(ALL[key]["confusion_matrix"]); labels = [l[:9] for l in ALL[key]["labels"]]
    ax.imshow(m, cmap="Blues")
    for (i, j), v in np.ndenumerate(m):
        ax.text(j, i, int(v), ha="center", va="center", color="white" if v > m.max() / 2 else "black")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right"); ax.set_yticks(range(len(labels)), labels)
    ax.set_title(key, fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("gold")
plt.tight_layout(); plt.savefig(OUT / "confusion_matrices.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(keys)); w = 0.38
ax.bar(x - w / 2, [ALL[k]["FAR"] for k in keys], w, label="FAR (lower is better)")
ax.bar(x + w / 2, [ALL[k]["paradox_identification_accuracy"] for k in keys], w, label="Paradox ID accuracy")
ax.set_xticks(x, keys, rotation=30, ha="right"); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "far_paradox.png", dpi=150); plt.show()

### Optional: publish results to the dataset repo
Writes `results/<date>/…` into `Mati83moni/mca-adversarial-benchmarks` (needs `HF_TOKEN` with write access).

In [ ]:
PUSH_RESULTS = False
if PUSH_RESULTS and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    stamp = time.strftime("%Y-%m-%d")
    HfApi().upload_folder(folder_path=str(OUT), path_in_repo=f"results/{stamp}", repo_id=DATASET_REPO,
                          repo_type="dataset", commit_message=f"Kaggle GPU evaluation {stamp}")